In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('/Users/gwan/Desktop/code/AI-Agent/LLM_AI_Agent/rag/data/OneNYC 2050 Strategic Plan.pdf')
data_nyc = loader.load()
print(data_nyc)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 데이터를 1000자로 단위로 나눔. Overlap은 100자로 설젇ㅇ
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
all_splits = text_splitter.split_documents(data_nyc)

In [ ]:
for i, split in enumerate(all_splits):
    print(f"split {i+1} --------------\n")
    print(split)

In [ ]:
print(type(all_splits[0]))

In [ ]:
loader_seoul = PyPDFLoader('/Users/gwan/Desktop/code/AI-Agent/LLM_AI_Agent/rag/data/Seoul 2040 Plan.pdf')
data_seoul = loader_seoul.load()
seoul_splits = text_splitter.split_documents(data_seoul)
for i, split in enumerate(seoul_splits):
    print(f"split {i+1}")
    print(split)

In [ ]:
print(seoul_splits[50].page_content)
print("---------")
print(seoul_splits[51].page_content)

In [ ]:
for i in range(len(seoul_splits) - 1):
    seoul_splits[i].page_content += "\n" + seoul_splits[i + 1].page_content[:100]

print(seoul_splits[50].page_content)
print("-------")
print(seoul_splits[51].page_content)

In [ ]:
print(len(all_splits))
all_splits.extend(seoul_splits)
print(len(all_splits))

In [ ]:
%pip install langchain_chroma

In [ ]:
from langchain_openai import OpenAIEmbeddings

from dotenv import laod_dotenv
import os

load_dotenv()

OPEN_AI_API = os.getenv('OPEN_API_KEY')
embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPEN_AI_API)
v = embedding.embed_query('뉴욕의 온실가스 저감 정책은 뭐야')
print(v)
print(len(v))

In [ ]:
from langchain_chroma import Chroma
import os

persist_directory = '../chroma_store'

if not os.path.exists(persist_directory):
    print("Creating new Chroma stroe")
    vectorstore = Chroma.from_documents(
        documens=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )
else:
    print("Loading Existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding,
    )
    

In [ ]:
retriever = vectorstore.as_retriever(k=3)
docs = retriever.invoke("서울시의 환경 정책이 궁금해")

for d in docs:
    print(d)
    print('-----')